In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
pulls_one = pd.read_csv('datasets/pulls_2011-2013.csv')
pulls_two = pd.read_csv('datasets/pulls_2014-2018.csv')
pull_files = pd.read_csv('datasets/pull_files.csv')

pulls = pd.concat([pulls_one, pulls_two])
pulls['date'] = pd.to_datetime(pulls['date'], utc=True)
data = pulls.merge(pull_files, on='pid')

In [ ]:
data['month'] = pd.DatetimeIndex(data['date']).month
data['year'] = pd.DatetimeIndex(data['date']).year
counts = data.groupby(['month', 'year']).count()
counts.plot(kind='bar', figsize=(12,4))

by_user = data.groupby('user').agg({'pid':'count'})
by_user.hist(bins=10)
plt.xlabel('Number of Contributions')
plt.ylabel('Number of Contributor')
plt.title('Is the project welcoming to the new Contributors ?')

last_10 = pulls[-10:]
joined_pr = last_10.merge(pull_files, on='pid')
files = set(joined_pr['file'].unique())
print(files)

In [ ]:
file = 'src/compiler/scala/reflect/reify/phases/Calculate.scala'
file_pr = data[data['file'] == file]
author_counts = file_pr['user'].value_counts().head(3)
print(author_counts)

file_pr = data[data['file'] == file]
joined_pr = file_pr.merge(pulls, on='pid')
users_last_10 = joined_pr.sort_values(by='date', ascending=False)['user'].head(10)
print(users_last_10)

authors = ['xeno-by', 'soc']
by_author = data[data['user'].isin(authors)]
by_author['year'] = by_author['date'].dt.year
counts = by_author.groupby(['year', 'user']).agg({'pid':'count'}).reset_index()
counts_wide = counts.pivot_table(index='year', columns='user', values='pid', fill_value=0)
counts_wide.plot(kind='bar')

by_author = data[data['user'].isin(authors)]
by_file = by_author[by_author['file'] == file]
grouped = by_file.groupby(['user', by_file['date'].dt.year]).count()['pid'].reset_index()
by_file_wide = grouped.pivot_table(index='date', columns='user', values='pid', fill_value=0)
by_file_wide.plot(kind='bar')